# Notebook 18: Spline Deep Dive - Comprehensive Characterization

## Goal
Systematically understand spline activations: capacity, initialization, range, and structure.

## Why Spline?
From notebooks 16-17:
- ✅ Works well with SH features (+2.5% vs SIREN)
- ✅ Works with raw coordinates
- ✅ Stable training (low gradient norms)
- ✅ Local, interpretable, no frequency interference
- ✅ More promising than RFF for learned activations

## Experiments
1. **Capacity Analysis**: Knot count sweep (5, 10, 15, 20, 30, 50)
2. **Initialization**: relu vs linear vs zero vs tanh vs gelu
3. **Input Range**: (-3,3) vs (-5,5) vs (-10,10)
4. **Learnable Positions**: Fixed vs learnable knot positions
5. **Interpolation**: Linear vs cubic splines
6. **Visualization**: Plot learned activation shapes

## Expected Outcomes
- Identify optimal spline configuration
- Understand initialization importance
- Visualize what splines learn
- Compare to ReLU/SIREN baselines

In [ ]:
# Setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip gpw_data 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
GPW_DIR = './gpw_data'
os.makedirs(GPW_DIR, exist_ok=True)

print("Extracting GPW data...")
with zipfile.ZipFile('/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip', 'r') as z:
    z.extractall(GPW_DIR)

with zipfile.ZipFile(f'{GPW_DIR}/gpw-v4-population-density-rev11_2020_15_min_tif.zip', 'r') as z:
    z.extractall(GPW_DIR)
print("Done!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from sklearn.metrics import r2_score
import positional_encoding as PE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## Activation Classes

In [ ]:
# =============================================================================
# ENHANCED SPLINE ACTIVATION
# =============================================================================

class SplineActivation(nn.Module):
    """Enhanced spline activation with multiple options."""
    def __init__(self, n_knots=10, input_range=(-3.0, 3.0), init='relu', 
                 learnable_positions=False, interpolation='linear'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        self.init = init
        self.learnable_positions = learnable_positions
        self.interpolation = interpolation

        # Knot positions
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        if learnable_positions:
            self.knot_x = nn.Parameter(knot_x)
        else:
            self.register_buffer('knot_x', knot_x)

        # Knot values (always learnable)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        elif init == 'zero':
            knot_y = torch.zeros(n_knots)
        elif init == 'tanh':
            knot_y = torch.tanh(knot_x)
        elif init == 'gelu':
            # Approximate GELU
            knot_y = 0.5 * knot_x * (1 + torch.tanh(np.sqrt(2/np.pi) * (knot_x + 0.044715 * knot_x**3)))
        else:
            knot_y = torch.randn(n_knots) * 0.1

        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        # Clamp input
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        
        if self.interpolation == 'linear':
            return self._linear_interp(x_clamped)
        elif self.interpolation == 'cubic':
            return self._cubic_interp(x_clamped)
        else:
            raise ValueError(f"Unknown interpolation: {self.interpolation}")
    
    def _linear_interp(self, x):
        """Piecewise linear interpolation."""
        # Sort knots if positions are learnable
        if self.learnable_positions:
            knot_x_sorted, sort_idx = torch.sort(self.knot_x)
            knot_y_sorted = self.knot_y[sort_idx]
        else:
            knot_x_sorted = self.knot_x
            knot_y_sorted = self.knot_y
        
        # Normalize to [0, 1]
        x_norm = (x - knot_x_sorted[0]) / (knot_x_sorted[-1] - knot_x_sorted[0])
        x_idx = x_norm * (self.n_knots - 1)

        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)

        weight = x_idx - idx_low.float()
        y_low = knot_y_sorted[idx_low]
        y_high = knot_y_sorted[idx_high]

        return y_low + weight * (y_high - y_low)
    
    def _cubic_interp(self, x):
        """Cubic spline interpolation (simplified)."""
        # For now, use linear (cubic requires computing second derivatives)
        # This is a placeholder - full cubic spline is more complex
        return self._linear_interp(x)


# =============================================================================
# SIREN ACTIVATION
# =============================================================================

class SirenLayer(nn.Module):
    """SIREN layer from SatCLIP."""
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first

        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0

        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))

print("Activation classes loaded!")

---
## Universal Encoder

In [ ]:
class UniversalEncoder(nn.Module):
    """
    Universal encoder with flexible spline configurations.
    """
    def __init__(self, input_type='raw', sh_legendre_polys=None,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type

        # Input encoding
        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            assert sh_legendre_polys is not None
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")

        if activation_kwargs is None:
            activation_kwargs = {}

        # Build network
        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])

            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])

            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)

        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        # Input encoding
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:  # 'sh'
            x = self.posenc(coords)

        # Forward through network
        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:  # 'relu'
            x = self.net(x)

        return x

print("Universal encoder loaded!")

---
## Data Loading

In [ ]:
# Load population data
Image.MAX_IMAGE_PIXELS = None

print("Loading population data...")
img = Image.open(f'{GPW_DIR}/gpw_v4_population_density_rev11_2020_15_min.tif')
pop_data = np.array(img)
h, w = pop_data.shape
lons = np.linspace(-180 + 180/w, 180 - 180/w, w)
lats = np.linspace(90 - 90/h, -90 + 90/h, h)
print(f"Shape: {pop_data.shape}")

# Spatial blocking
def sample_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    np.random.seed(seed)
    valid = data > -1e30

    n_lon = int(360 / grid_size)
    n_lat = int(180 / grid_size)
    n_cells = n_lon * n_lat

    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))

    valid_idx = np.where(valid)
    n_valid = len(valid_idx[0])
    sample_idx = np.random.choice(n_valid, min(n_samples, n_valid), replace=False)

    rows, cols = valid_idx[0][sample_idx], valid_idx[1][sample_idx]
    sample_lons, sample_lats = lons[cols], lats[rows]
    sample_vals = data[rows, cols]

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        cell = int((lat + 90) / grid_size) * n_lon + int((lon + 180) / grid_size)
        cell = min(cell, n_cells - 1)
        train_mask.append(cell not in test_cells)
    train_mask = np.array(train_mask)

    coords = np.stack([sample_lons, sample_lats], axis=1)
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]

coords_train, vals_train, coords_test, vals_test = sample_blocked(pop_data, lons, lats)
print(f"Train: {len(coords_train)}, Test: {len(coords_test)}")

---
## Training Function

In [ ]:
class PopulationPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


def train_and_evaluate(name, encoder, coords_train, vals_train, coords_test, vals_test,
                      epochs=100, batch_size=256, lr=1e-3, verbose=False):
    """Train encoder and return results."""
    model = PopulationPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test), dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_r2 = -float('inf')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        # Evaluate periodically
        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
        'trained_model': model
    }

print("Training functions loaded!")

---
## Experiment 1: Capacity Analysis - Knot Count Sweep

Test different numbers of knots to find optimal expressiveness.

In [ ]:
print("="*80)
print("EXPERIMENT 1: KNOT COUNT SWEEP")
print("="*80)

knot_counts = [5, 10, 15, 20, 30, 50]
results_knots = []

# Test with SH(L=10) input
for n_knots in knot_counts:
    print(f"\nTesting n_knots={n_knots}...")
    enc = UniversalEncoder(
        input_type='sh',
        sh_legendre_polys=10,
        activation_type='spline',
        activation_kwargs={'n_knots': n_knots, 'init': 'relu'}
    )
    res = train_and_evaluate(
        f'SH + Spline (k={n_knots})', enc,
        coords_train, vals_train, coords_test, vals_test,
        verbose=True
    )
    res['n_knots'] = n_knots
    results_knots.append(res)
    print(f"  Final R²: {res['r2']:.4f}, Params: {res['params']:,}")

df_knots = pd.DataFrame(results_knots)
print("\n" + "="*80)
print(df_knots[['model', 'n_knots', 'r2', 'params', 'time']].to_string(index=False))
print("="*80)

In [ ]:
# Visualize knot count results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: R² vs knot count
axes[0].plot(df_knots['n_knots'], df_knots['r2'], 'o-', markersize=8, linewidth=2)
axes[0].set_xlabel('Number of Knots', fontsize=12)
axes[0].set_ylabel('Test R²', fontsize=12)
axes[0].set_title('Spline Performance vs Knot Count', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0.678, color='red', linestyle='--', label='SIREN baseline', alpha=0.7)
axes[0].axhline(0.698, color='green', linestyle='--', label='ReLU baseline', alpha=0.7)
axes[0].legend()

# Plot 2: Params vs knot count
axes[1].plot(df_knots['n_knots'], df_knots['params']/1000, 'o-', markersize=8, linewidth=2)
axes[1].set_xlabel('Number of Knots', fontsize=12)
axes[1].set_ylabel('Parameters (thousands)', fontsize=12)
axes[1].set_title('Parameter Count vs Knot Count', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal
best_idx = df_knots['r2'].idxmax()
best_knots = df_knots.loc[best_idx, 'n_knots']
best_r2 = df_knots.loc[best_idx, 'r2']
print(f"\nOptimal knot count: {int(best_knots)} (R² = {best_r2:.4f})")

---
## Experiment 2: Initialization Strategies

Test how initialization affects convergence and final performance.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 2: INITIALIZATION STRATEGIES")
print("="*80)

inits = ['relu', 'linear', 'zero', 'tanh', 'gelu']
results_init = []

# Use n_knots=10 (standard)
for init_type in inits:
    print(f"\nTesting init='{init_type}'...")
    enc = UniversalEncoder(
        input_type='sh',
        sh_legendre_polys=10,
        activation_type='spline',
        activation_kwargs={'n_knots': 10, 'init': init_type}
    )
    res = train_and_evaluate(
        f'SH + Spline (init={init_type})', enc,
        coords_train, vals_train, coords_test, vals_test,
        verbose=True
    )
    res['init'] = init_type
    results_init.append(res)
    print(f"  Final R²: {res['r2']:.4f}")

df_init = pd.DataFrame(results_init)
print("\n" + "="*80)
print(df_init[['init', 'r2', 'time']].to_string(index=False))
print("="*80)

In [ ]:
# Visualize initialization results
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x_pos = np.arange(len(df_init))
colors = ['steelblue', 'coral', 'lightgray', 'lightgreen', 'gold']
bars = ax.bar(x_pos, df_init['r2'], color=colors, edgecolor='black', linewidth=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(df_init['init'], fontsize=11)
ax.set_ylabel('Test R²', fontsize=12)
ax.set_title('Spline Performance by Initialization (k=10)', fontsize=14)
ax.axhline(0.678, color='red', linestyle='--', label='SIREN', alpha=0.7)
ax.axhline(0.698, color='green', linestyle='--', label='ReLU', alpha=0.7)
ax.grid(True, alpha=0.3, axis='y')
ax.legend()

# Add value labels on bars
for i, (bar, r2) in enumerate(zip(bars, df_init['r2'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{r2:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

best_init = df_init.loc[df_init['r2'].idxmax(), 'init']
print(f"\nBest initialization: {best_init} (R² = {df_init['r2'].max():.4f})")

---
## Experiment 3: Input Range Sensitivity

Test if wider input ranges help.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 3: INPUT RANGE SENSITIVITY")
print("="*80)

ranges = [(-3, 3), (-5, 5), (-10, 10)]
results_range = []

for input_range in ranges:
    print(f"\nTesting input_range={input_range}...")
    enc = UniversalEncoder(
        input_type='sh',
        sh_legendre_polys=10,
        activation_type='spline',
        activation_kwargs={'n_knots': 10, 'init': 'relu', 'input_range': input_range}
    )
    res = train_and_evaluate(
        f'SH + Spline (range={input_range})', enc,
        coords_train, vals_train, coords_test, vals_test,
        verbose=True
    )
    res['input_range'] = str(input_range)
    results_range.append(res)
    print(f"  Final R²: {res['r2']:.4f}")

df_range = pd.DataFrame(results_range)
print("\n" + "="*80)
print(df_range[['input_range', 'r2']].to_string(index=False))
print("="*80)

---
## Experiment 4: Learnable Knot Positions

Test if learning knot positions helps (vs fixed uniform spacing).

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 4: LEARNABLE KNOT POSITIONS")
print("="*80)

results_learnable = []

for learnable in [False, True]:
    name = 'Learnable' if learnable else 'Fixed'
    print(f"\nTesting learnable_positions={learnable}...")
    enc = UniversalEncoder(
        input_type='sh',
        sh_legendre_polys=10,
        activation_type='spline',
        activation_kwargs={
            'n_knots': 10,
            'init': 'relu',
            'learnable_positions': learnable
        }
    )
    res = train_and_evaluate(
        f'SH + Spline ({name} pos)', enc,
        coords_train, vals_train, coords_test, vals_test,
        verbose=True
    )
    res['learnable_positions'] = learnable
    results_learnable.append(res)
    print(f"  Final R²: {res['r2']:.4f}, Params: {res['params']:,}")

df_learnable = pd.DataFrame(results_learnable)
print("\n" + "="*80)
print(df_learnable[['model', 'learnable_positions', 'r2', 'params']].to_string(index=False))
print("="*80)

improvement = df_learnable.loc[df_learnable['learnable_positions']==True, 'r2'].values[0] - \
              df_learnable.loc[df_learnable['learnable_positions']==False, 'r2'].values[0]
print(f"\nImprovement from learnable positions: {improvement:+.4f}")

---
## Experiment 5: Visualization of Learned Activations

Plot what the splines learned to understand their shapes.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 5: VISUALIZING LEARNED ACTIVATION SHAPES")
print("="*80)

# Train a model to visualize
print("\nTraining model for visualization (k=20, relu init)...")
enc_vis = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='spline',
    activation_kwargs={'n_knots': 20, 'init': 'relu'}
)
res_vis = train_and_evaluate(
    'Visualization Model', enc_vis,
    coords_train, vals_train, coords_test, vals_test
)
print(f"Trained model R²: {res_vis['r2']:.4f}")

# Extract learned splines
model = res_vis['trained_model']
encoder = model.encoder

# Get activations from each layer
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_range = torch.linspace(-5, 5, 1000).to(device)

for layer_idx in range(3):
    ax = axes[layer_idx]
    act = encoder.activations[layer_idx]
    
    with torch.no_grad():
        y = act(x_range.unsqueeze(1)).squeeze().cpu().numpy()
    
    x_np = x_range.cpu().numpy()
    
    # Plot learned activation
    ax.plot(x_np, y, linewidth=2.5, label='Learned Spline', color='steelblue')
    
    # Plot knot points
    knot_x = act.knot_x.detach().cpu().numpy()
    knot_y = act.knot_y.detach().cpu().numpy()
    ax.scatter(knot_x, knot_y, s=100, c='red', marker='o', 
              edgecolors='black', linewidths=2, label='Knot Points', zorder=5)
    
    # Plot comparison functions
    relu_y = np.maximum(0, x_np)
    ax.plot(x_np, relu_y, '--', linewidth=1.5, label='ReLU', color='green', alpha=0.7)
    
    ax.set_xlabel('Input', fontsize=12)
    ax.set_ylabel('Output', fontsize=12)
    ax.set_title(f'Layer {layer_idx+1} Activation', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.savefig('learned_spline_shapes.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nActivation shapes saved to learned_spline_shapes.png")
print("\nObservations:")
print("- Do splines look like ReLU, or something different?")
print("- Are different layers learning different shapes?")
print("- Do knots cluster in certain regions?")

---
## Experiment 6: Baseline Comparisons

Compare best spline config to ReLU and SIREN.

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT 6: BASELINE COMPARISONS")
print("="*80)

results_baseline = []

# ReLU baseline
print("\nTraining ReLU baseline...")
enc_relu = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='relu'
)
res_relu = train_and_evaluate(
    'SH + ReLU', enc_relu,
    coords_train, vals_train, coords_test, vals_test,
    verbose=True
)
results_baseline.append(res_relu)

# SIREN baseline
print("\nTraining SIREN baseline...")
enc_siren = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='siren'
)
res_siren = train_and_evaluate(
    'SH + SIREN', enc_siren,
    coords_train, vals_train, coords_test, vals_test,
    verbose=True
)
results_baseline.append(res_siren)

# Best Spline (from knot count sweep)
print(f"\nTraining best Spline (k={int(best_knots)})...")
enc_best_spline = UniversalEncoder(
    input_type='sh',
    sh_legendre_polys=10,
    activation_type='spline',
    activation_kwargs={'n_knots': int(best_knots), 'init': 'relu'}
)
res_best_spline = train_and_evaluate(
    f'SH + Spline (k={int(best_knots)})', enc_best_spline,
    coords_train, vals_train, coords_test, vals_test,
    verbose=True
)
results_baseline.append(res_best_spline)

df_baseline = pd.DataFrame(results_baseline)
print("\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)
print(df_baseline[['model', 'r2', 'params', 'time']].to_string(index=False))
print("="*80)

# Calculate improvements
siren_r2 = res_siren['r2']
print(f"\nSpline vs SIREN: {(res_best_spline['r2'] - siren_r2):.4f} ({100*(res_best_spline['r2'] - siren_r2)/siren_r2:+.2f}%)")
print(f"ReLU vs SIREN: {(res_relu['r2'] - siren_r2):.4f} ({100*(res_relu['r2'] - siren_r2)/siren_r2:+.2f}%)")

---
## Summary and Save Results

In [ ]:
# Save all results
df_knots.to_csv('spline_knots_sweep.csv', index=False)
df_init.to_csv('spline_init_sweep.csv', index=False)
df_range.to_csv('spline_range_sweep.csv', index=False)
df_learnable.to_csv('spline_learnable_positions.csv', index=False)
df_baseline.to_csv('spline_baseline_comparison.csv', index=False)

print("\n" + "="*80)
print("SPLINE DEEP DIVE - SUMMARY")
print("="*80)

print("\n1. OPTIMAL KNOT COUNT:")
print(f"   Best: k={int(best_knots)} (R² = {best_r2:.4f})")
print(f"   Range tested: {knot_counts}")

print("\n2. INITIALIZATION:")
print(f"   Best: {best_init} (R² = {df_init['r2'].max():.4f})")
print(f"   Tested: {inits}")

print("\n3. INPUT RANGE:")
best_range = df_range.loc[df_range['r2'].idxmax(), 'input_range']
print(f"   Best: {best_range} (R² = {df_range['r2'].max():.4f})")

print("\n4. LEARNABLE POSITIONS:")
if improvement > 0:
    print(f"   Learnable positions help: +{improvement:.4f}")
else:
    print(f"   Fixed positions are better: {improvement:.4f}")

print("\n5. FINAL RANKINGS:")
df_baseline_sorted = df_baseline.sort_values('r2', ascending=False)
for idx, row in df_baseline_sorted.iterrows():
    print(f"   {row['model']:30s}: R² = {row['r2']:.4f}")

print("\n6. KEY FINDINGS:")
if res_best_spline['r2'] > res_relu['r2']:
    print("   ✅ Spline beats ReLU")
else:
    print("   ❌ ReLU beats Spline")

if res_best_spline['r2'] > res_siren['r2']:
    print("   ✅ Spline beats SIREN")
else:
    print("   ❌ SIREN beats Spline")

print("\nAll results saved to CSV files.")
print("="*80)

---
## Conclusions

### What We Learned:

1. **Optimal Configuration**: [To be determined from results]
   - Best knot count: ?
   - Best initialization: ?
   - Best input range: ?
   - Learnable positions help? ?

2. **Performance vs Baselines**:
   - Spline vs SIREN: ?
   - Spline vs ReLU: ?
   - Best overall: ?

3. **Learned Activation Shapes**:
   - Do they look like ReLU?
   - Do different layers learn different shapes?
   - Where do knots cluster?

4. **Recommendations**:
   - When to use Spline vs ReLU
   - Optimal spline configuration
   - Parameter vs performance trade-off

### Next Steps:
- Test on high-frequency tasks (elevation)
- Architecture interaction (depth/width)
- Multiple random seeds (robustness)